# Day 5: Neural Networks & ANN (Keras)

This notebook trains a feedforward artificial neural network to predict whether a Telco customer will churn. It keeps the test set separate until final evaluation and saves everything needed for future predictions.

## ANN overview

A feedforward ANN passes input features through fully connected hidden layers. ReLU activation helps hidden layers learn non-linear patterns. The final Sigmoid unit produces a churn probability between 0 and 1. Binary cross-entropy compares that probability with the observed churn label.

In [ ]:
from pathlib import Path
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow import keras

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists():
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODELS_DIR / 'day5_ann_churn_model.keras'
PREPROCESSOR_PATH = MODELS_DIR / 'day5_ann_preprocessor.joblib'


## Load and prepare the data

`TotalCharges` is converted to numeric because blank entries represent missing values. Customer IDs are identifiers rather than customer-behavior signals, so they are excluded. Missing values, encoding, and scaling are learned from training data only.

In [ ]:
df = pd.read_csv(DATA_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train).astype('float32')
X_test_processed = preprocessor.transform(X_test).astype('float32')

print(f'Training rows: {X_train_processed.shape[0]}')
print(f'Test rows: {X_test_processed.shape[0]}')
print(f'Prepared features: {X_train_processed.shape[1]}')
print(f'Churn rate: {y.mean():.1%}')

## Build and train the ANN

The input size equals the number of preprocessed features. Two ReLU hidden layers learn patterns, while dropout randomly disables a small fraction of training connections to reduce overfitting. Adam adapts the learning rate during optimization. Early stopping watches validation loss and restores the best weights.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_processed.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.20),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dropout(0.10),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=12, restore_best_weights=True
)

history = model.fit(
    X_train_processed, y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print(f'Best epoch: {np.argmin(history.history["val_loss"]) + 1}')

## Learning curves

Training and validation lines that improve together suggest useful learning. A widening gap where training loss falls while validation loss rises is a sign of overfitting.

In [ ]:
epochs = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history.history['accuracy'], label='Training accuracy')
axes[0].plot(epochs, history.history['val_accuracy'], label='Validation accuracy')
axes[0].set(title='Model accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[0].legend()

axes[1].plot(epochs, history.history['loss'], label='Training loss')
axes[1].plot(epochs, history.history['val_loss'], label='Validation loss')
axes[1].set(title='Model loss', xlabel='Epoch', ylabel='Binary cross-entropy')
axes[1].legend()

plt.tight_layout()
plt.show()

best_epoch_index = int(np.argmin(history.history['val_loss']))
accuracy_gap = history.history['accuracy'][best_epoch_index] - history.history['val_accuracy'][best_epoch_index]
if accuracy_gap > 0.08:
    print(f'At the best validation-loss epoch, the training-validation accuracy gap is {accuracy_gap:.1%}. This suggests some overfitting.')
else:
    print(f'At the best validation-loss epoch, the training-validation accuracy gap is {accuracy_gap:.1%}. The curves suggest limited overfitting.')

## Test-set evaluation

A probability of at least 0.50 is classified as churn. Accuracy gives overall correctness; precision measures how reliable positive churn predictions are; recall measures how much observed churn is found; F1 balances precision and recall.

In [ ]:
y_probability = model.predict(X_test_processed, verbose=0).ravel()
y_pred = (y_probability >= 0.50).astype(int)

precision, recall, f1_score, _ = precision_recall_fscore_support(
    y_test, y_pred, average='binary', zero_division=0
)
metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision,
    'Recall': recall,
    'F1-score': f1_score
})
display(metrics.to_frame('Score').style.format('{:.3f}'))

print(classification_report(y_test, y_pred, target_names=['No churn', 'Churn'], zero_division=0))

confusion = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=['No churn', 'Churn']).plot(cmap='Blues')
plt.title('ANN confusion matrix')
plt.show()

## Save and reuse the model

The Keras model and fitted preprocessor are saved separately. To score new raw customer records, remove `customerID` and `Churn` if present, transform the remaining columns with the saved preprocessor, and pass the result to the loaded model.

In [ ]:
model.save(MODEL_PATH)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

loaded_model = keras.models.load_model(MODEL_PATH)
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

new_customer = X_test.iloc[[0]].copy()
new_customer_processed = loaded_preprocessor.transform(new_customer).astype('float32')
new_customer_probability = float(loaded_model.predict(new_customer_processed, verbose=0)[0][0])

print(f'Model saved to: {MODEL_PATH}')
print(f'Preprocessor saved to: {PREPROCESSOR_PATH}')
print(f'Sample churn probability: {new_customer_probability:.1%}')
print(f'Sample prediction: {"Churn" if new_customer_probability >= 0.50 else "No churn"}')